# 🧬 Self-Replicating Coding Agent — Colab Backend

Runs the full evolution pipeline locally on T4 GPU using **Qwen3-14B** via Ollama.
No Groq/Gemini rate limits. Results saved to Google Drive.

| | |
|---|---|
| **Hardware** | T4 GPU (~2 Colab units/hr) |
| **Credits** | 300 units → **~150 hours** of evolution |
| **Model** | `qwen3:14b` — Q4, ~9 GB VRAM, built-in reasoning mode |
| **Mode** | `/no_think` for fast coding tasks, `/think` for complex reasoning |

---
### Run order
**Cell 8** (keep-alive) → **Cell 1** → **Cell 2** → **Cell 3** (one-time, ~5 min) → **Cell 4** → **Cell 5** → **Cell 6** 🚀

In [ ]:
#@title ⏰ Cell 8 — Keep-alive (run FIRST before long cells)
#@markdown Prevents Colab idle timeout. Run this, then proceed with Cell 1.

import threading, time, datetime

_keep_alive = True

def _heartbeat():
    start = datetime.datetime.now()
    while _keep_alive:
        elapsed = datetime.datetime.now() - start
        hrs, rem = divmod(int(elapsed.total_seconds()), 3600)
        mins, secs = divmod(rem, 60)
        print(f'💓 {datetime.datetime.now().strftime("%H:%M:%S")} — alive {hrs:02d}:{mins:02d}:{secs:02d}', end='\r')
        time.sleep(30)

t = threading.Thread(target=_heartbeat, daemon=True)
t.start()
print('✅ Keep-alive started — heartbeat every 30s (shows elapsed time)')
print('   To stop: _keep_alive = False')

In [ ]:
#@title 📂 Cell 1 — Mount Google Drive & set up project

from google.colab import drive
import os, shutil, subprocess

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/SelfReplicatingAgent'
WORK_DIR  = '/content/SelfReplicatingAgent'
os.makedirs(DRIVE_DIR, exist_ok=True)

# ── Option A: zip uploaded to Drive ─────────────────────────────────────────
zip_path = f'{DRIVE_DIR}/2ndRunSelfReplicatingAgent.zip'
if os.path.exists(zip_path) and not os.path.exists(WORK_DIR):
    print('📦 Extracting zip from Drive...')
    shutil.unpack_archive(zip_path, '/content/')
    extracted = [d for d in os.listdir('/content/')
                 if 'SelfReplicating' in d and os.path.isdir(f'/content/{d}')]
    if extracted and extracted[0] != 'SelfReplicatingAgent':
        os.rename(f'/content/{extracted[0]}', WORK_DIR)
    print(f'✅ Extracted to {WORK_DIR}')

# ── Option B: clone from HuggingFace ────────────────────────────────────────
elif not os.path.exists(WORK_DIR):
    print('🔗 Cloning from HuggingFace...')
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://huggingface.co/spaces/Balaji33k/self-replicating-agent', WORK_DIR],
        check=True
    )
    print(f'✅ Cloned to {WORK_DIR}')
else:
    print(f'✅ Project already at {WORK_DIR}')

# ── Symlink data/ → Drive (results persist across reconnects) ────────────────
DATA_DRIVE = f'{DRIVE_DIR}/data'
DATA_LOCAL = f'{WORK_DIR}/data'
os.makedirs(DATA_DRIVE, exist_ok=True)
if os.path.exists(DATA_LOCAL) and not os.path.islink(DATA_LOCAL):
    for f in os.listdir(DATA_LOCAL):
        dst = f'{DATA_DRIVE}/{f}'
        if not os.path.exists(dst):
            shutil.copy2(f'{DATA_LOCAL}/{f}', dst)
    shutil.rmtree(DATA_LOCAL)
if not os.path.islink(DATA_LOCAL):
    os.symlink(DATA_DRIVE, DATA_LOCAL)
    print(f'🔗 data/ → Drive')

print('✅ Ready')
print('Files:', os.listdir(WORK_DIR))

In [ ]:
#@title 📦 Cell 2 — Install Python dependencies

import subprocess, sys
WORK_DIR = '/content/SelfReplicatingAgent'

print('Installing...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{WORK_DIR}/requirements.txt'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('ERROR:', r.stderr[-2000:])
else:
    print('✅ Dependencies installed')

for pkg in ['langchain_groq', 'langchain_google_genai', 'langchain_openai']:
    try:
        __import__(pkg); print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} MISSING')

In [ ]:
#@title 🦙 Cell 3 — Install Ollama + pull Qwen3-14B
#@markdown Downloads ~9 GB once, cached to Drive → instant on reconnect.
#@markdown First run takes ~4-6 minutes.

import subprocess, os, shutil, time

DRIVE_DIR    = '/content/drive/MyDrive/SelfReplicatingAgent'
MODEL_CACHE  = f'{DRIVE_DIR}/ollama_models'
OLLAMA_MODEL = 'qwen3:14b'  #@param ["qwen3:14b", "qwen3:8b", "qwen3:30b-a3b", "qwen2.5-coder:14b"]
#@markdown > **qwen3:14b** — 14B params, thinking mode, ~9 GB VRAM (fits T4)  
#@markdown > **qwen3:30b-a3b** — MoE: 30B quality at 3B speed (~20 GB, needs L4/A100)  
#@markdown > **qwen3:8b** — faster, lower quality, ~5 GB VRAM  

os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['OLLAMA_MODELS'] = MODEL_CACHE

# Install Ollama
if not shutil.which('ollama'):
    print('Installing Ollama...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, check=True)
    print('✅ Ollama installed')
else:
    print('✅ Ollama already installed')

# Check cache
model_slug = OLLAMA_MODEL.replace(':', '/')
manifest_path = os.path.join(MODEL_CACHE, 'manifests', 'registry.ollama.ai',
                              'library', model_slug)
if os.path.exists(manifest_path):
    print(f'✅ {OLLAMA_MODEL} already in Drive cache — skipping download')
else:
    size_hint = {'qwen3:14b': '~9 GB', 'qwen3:8b': '~5 GB',
                 'qwen3:30b-a3b': '~20 GB', 'qwen2.5-coder:14b': '~9 GB'}
    print(f'⬇️  Pulling {OLLAMA_MODEL} ({size_hint.get(OLLAMA_MODEL, "?")} — please wait)...')

    srv = subprocess.Popen(
        ['ollama', 'serve'],
        env={**os.environ, 'OLLAMA_MODELS': MODEL_CACHE},
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(3)

    pull = subprocess.run(
        ['ollama', 'pull', OLLAMA_MODEL],
        env={**os.environ, 'OLLAMA_MODELS': MODEL_CACHE},
        capture_output=True, text=True
    )
    srv.terminate()

    if pull.returncode != 0:
        print('❌ Pull failed:', pull.stderr[-1000:])
    else:
        print(f'✅ {OLLAMA_MODEL} downloaded and cached to Drive')

with open('/content/ollama_model.txt', 'w') as f:
    f.write(OLLAMA_MODEL)
print(f'📝 Model set: {OLLAMA_MODEL}')

In [ ]:
#@title 🚀 Cell 4 — Start Ollama server & verify

import subprocess, os, time, urllib.request, json

DRIVE_DIR   = '/content/drive/MyDrive/SelfReplicatingAgent'
MODEL_CACHE = f'{DRIVE_DIR}/ollama_models'
OLLAMA_MODEL = open('/content/ollama_model.txt').read().strip()

# Kill any previous instance
subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)

# Start server
env = {**os.environ, 'OLLAMA_MODELS': MODEL_CACHE}
subprocess.Popen(
    ['ollama', 'serve'], env=env,
    stdout=open('/content/ollama.log', 'w'),
    stderr=subprocess.STDOUT
)

print('Waiting for Ollama...')
for i in range(25):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print(f'✅ Server ready ({i+1}s)')
        break
    except Exception:
        time.sleep(1)
        print(f'  {i+1}/25...', end='\r')
else:
    print('❌ Server failed — check /content/ollama.log')
    print(open('/content/ollama.log').read()[-2000:])

# List loaded models
r = subprocess.run(['ollama', 'list'], capture_output=True, text=True, env=env)
print('Models:\n', r.stdout)

# Set env vars for llm_client.py
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']    = OLLAMA_MODEL
print(f'🎯 Provider: Ollama  Model: {OLLAMA_MODEL}')

# Smoke test — use /no_think for fast response (Qwen3 specific)
print('Running smoke test...')
try:
    req = urllib.request.Request(
        'http://localhost:11434/api/generate',
        data=json.dumps({
            'model': OLLAMA_MODEL,
            'prompt': '/no_think Write a Python hello world function.',
            'stream': False
        }).encode(),
        headers={'Content-Type': 'application/json'}
    )
    resp = json.loads(urllib.request.urlopen(req, timeout=90).read())
    reply = resp['response'].strip()[:200]
    print(f'🧪 Response: {reply}')
    print('✅ Qwen3-14B is working!')
except Exception as e:
    print(f'⚠️  Smoke test failed: {e}')

In [ ]:
#@title 🔑 Cell 5 — Optional API fallback keys
#@markdown Only used if Ollama goes down. Add to Colab Secrets panel for safety.

import os
from google.colab import userdata

def _secret(key):
    try: return userdata.get(key) or ''
    except: return ''

groq_key   = _secret('GROQ_API_KEY')
gemini_key = _secret('GEMINI_API_KEY')

if groq_key:   os.environ['GROQ_API_KEY']   = groq_key;   print('✅ GROQ_API_KEY loaded')
if gemini_key: os.environ['GEMINI_API_KEY'] = gemini_key; print('✅ GEMINI_API_KEY loaded')

provider = (
    '🦙 Ollama — qwen3:14b (primary, no rate limits)'  if os.environ.get('OLLAMA_BASE_URL') else
    '⚡ Groq (rate limits apply)'                      if groq_key else
    '✨ Gemini (rate limits apply)'                    if gemini_key else
    '❌ NO PROVIDER — run Cell 4 first'
)
print(f'Provider: {provider}')

In [ ]:
#@title 🧬 Cell 6 — Run Evolution Pipeline
#@markdown Each generation: 20 tasks → analyse failures → spawn next gen.
#@markdown Results auto-saved to Drive every task.

import subprocess, sys, os, json
from pathlib import Path

WORK_DIR  = '/content/SelfReplicatingAgent'
DATA_DIR  = f'{WORK_DIR}/data'
START_GEN = 1  #@param {type:"integer"}
MAX_GENS  = 10 #@param {type:"integer"}

os.makedirs(DATA_DIR, exist_ok=True)

# Clear stop flag
stop_flag = Path(f'{DATA_DIR}/stop.flag')
if stop_flag.exists():
    stop_flag.unlink()
    print('🗑️  Cleared stop.flag')

print(f'🚀 Evolution: Gen {START_GEN} → Gen {START_GEN + MAX_GENS - 1}')
print(f'🦙 Model: {os.environ.get("OLLAMA_MODEL", "(API fallback)")}')
print(f'⏱️  ~2 hrs/gen on T4 → {MAX_GENS * 2} hrs = ~{MAX_GENS * 2 * 2:.0f} Colab units')
print('=' * 60)

gen_dir = Path(WORK_DIR) / 'generations' / f'gen_{START_GEN}'
if not gen_dir.exists():
    raise FileNotFoundError(f'Gen {START_GEN} not found: {gen_dir}')

log_path = f'{DATA_DIR}/colab_run.log'
log_file = open(log_path, 'w', buffering=1)

proc = subprocess.Popen(
    [sys.executable, 'main.py', '--gen', str(START_GEN)],
    cwd=str(gen_dir),
    env={**os.environ},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
print(f'▶ PID={proc.pid}  Log: {log_path}')

try:
    for line in proc.stdout:
        line = line.rstrip()
        if line:
            print(line)
            log_file.write(line + '\n')
except KeyboardInterrupt:
    print('\n⏹ Stopped')
    proc.terminate()
finally:
    log_file.close()

ret = proc.wait()
print(f'\n✅ Finished (exit {ret})')

# Quick summary
results_dir = gen_dir / 'results'
if results_dir.exists():
    passed = failed = 0
    for f in results_dir.glob('*.json'):
        try:
            d = json.loads(f.read_text())
            if d.get('status') == 'success': passed += 1
            else: failed += 1
        except: pass
    total = passed + failed
    if total:
        bar = '█' * passed + '░' * failed
        print(f'📊 Gen {START_GEN}: [{bar[:20]}] {passed/total*100:.1f}% ({passed}/{total})')

In [ ]:
#@title 📊 Cell 7 — Results Dashboard

import json
from pathlib import Path

WORK_DIR = '/content/SelfReplicatingAgent'
gens_dir = Path(WORK_DIR) / 'generations'

print('=' * 60)
print('  EVOLUTION RESULTS')
print('=' * 60)

total_passed = total_failed = 0
for gen_dir in sorted(gens_dir.iterdir()):
    if not gen_dir.name.startswith('gen_'): continue
    results_dir = gen_dir / 'results'
    if not results_dir.exists(): continue

    passed = failed = 0
    error_types = {}
    for f in results_dir.glob('*.json'):
        try:
            d = json.loads(f.read_text())
            if d.get('status') == 'success':
                passed += 1
            else:
                failed += 1
                err = str(d.get('error_type', d.get('error', 'unknown')))[:35]
                error_types[err] = error_types.get(err, 0) + 1
        except: pass

    total = passed + failed
    if total == 0: continue
    pct = passed / total * 100
    bar = '█' * passed + '░' * failed
    arrow = '📈' if pct >= 50 else '📉' if pct < 25 else '➡️'

    print(f'\n{arrow} {gen_dir.name.upper():8s} [{bar[:20]:20s}] {pct:5.1f}%  ({passed}/{total})')
    for err, cnt in sorted(error_types.items(), key=lambda x: -x[1])[:3]:
        print(f'           └─ {err}: {cnt}x')

    total_passed += passed
    total_failed += failed

overall = total_passed + total_failed
if overall:
    print(f'\n{"="*60}')
    print(f'Overall: {total_passed/overall*100:.1f}% ({total_passed}/{overall} tasks)')

# Credits used estimate
import os
log = Path('/content/ollama.log')
if log.exists():
    lines = log.read_text().splitlines()[-3:]
    print('\n🦙 Ollama:', ' | '.join(l.strip() for l in lines if l.strip()))